In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
cd drive/MyDrive/house_price_prediction/

/content/drive/MyDrive/house_price_prediction


In [3]:
ls -l

total 312
-rw------- 1 root root 302183 Jul 10 02:33 house.csv
-rw------- 1 root root  15852 Jul 10 07:16 house_price_model.ipynb
-rw------- 1 root root    183 Jul 10 07:06 readme.md.gdoc


In [4]:
import pandas as pd

df = pd.read_csv("house.csv")
df.head()

,House_ID,Area_sqft,Bedrooms,Bathrooms,Floors,House_Age,Garage,Parking,Location,Furnished,Garden,Condition,Distance_to_City_km,Price
0,H00001,1028,3,1,2,17,0,0,Suburban,No,No,Excellent,5.6,351659
1,H00002,3334,6,2,1,0,3,0,Rural,No,Yes,Excellent,10.1,807321
2,H00003,1435,1,1,3,9,1,0,Suburban,No,No,Average,14.6,340628
3,H00004,3099,3,5,2,13,2,0,Suburban,No,No,Needs Renovation,11.9,675886
4,H00005,2086,6,1,2,16,1,2,Suburban,No,Yes,Needs Renovation,12.2,527347


In [18]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   House_ID             5000 non-null   object 
 1   Area_sqft            5000 non-null   int64  
 2   Bedrooms             5000 non-null   int64  
 3   Bathrooms            5000 non-null   int64  
 4   Floors               5000 non-null   int64  
 5   House_Age            5000 non-null   int64  
 6   Garage               5000 non-null   int64  
 7   Parking              5000 non-null   int64  
 8   Location             5000 non-null   object 
 9   Furnished            5000 non-null   object 
 10  Garden               5000 non-null   object 
 11  Condition            5000 non-null   object 
 12  Distance_to_City_km  5000 non-null   float64
 13  Price                5000 non-null   int64  
dtypes: float64(1), int64(8), object(5)
memory usage: 547.0+ KB


In [5]:
X = df[['Area_sqft',
        'Bedrooms',
        'Bathrooms',
        'Floors',
        'House_Age',
        'Garage',
        'Parking',
        'Location',
        'Furnished',
        'Garden',
        'Condition',
        'Distance_to_City_km',]]

y = df['Price']

In [6]:
from sklearn.preprocessing import LabelEncoder

# Encode all categorical columns
label_encoders = {}

for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

/tmp/ipykernel_1117/1077961541.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = le.fit_transform(X[col].astype(str))
/tmp/ipykernel_1117/1077961541.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = le.fit_transform(X[col].astype(str))
/tmp/ipykernel_1117/1077961541.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pand

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)
print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 4000
Testing Samples: 1000


In [8]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    random_state=42
)

model

RandomForestRegressor(random_state=42)

In [9]:
print(model)

RandomForestRegressor(random_state=42)


In [10]:
model.fit(X_train, y_train)
print("Model Training Completed.")


Model Training Completed.


In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predict
y_pred = model.predict(X_test)

# Evaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Model Evaluation")
print("-------------------------")
print("Mean Absolute Error (MAE):", mae)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("R² Score:", r2)


Model Evaluation
-------------------------
Mean Absolute Error (MAE): 24371.772100000002
Mean Squared Error (MSE): 922231547.0567272
Root Mean Squared Error (RMSE): 30368.265460126746
R² Score: 0.9841335515043617


In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

params = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 10, None]
}

grid = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=params,
    cv=3,
    scoring="r2"
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

print("Best Parameters:", grid.best_params_)
print("Best R² Score:", grid.best_score_)

Best Parameters: {'max_depth': None, 'n_estimators': 200}
Best R² Score: 0.9826146514752822


In [19]:
import joblib

joblib.dump(
    best_model,
    "house_price_model.pkl"
)

print("Model Saved Successfully.")

Model Saved Successfully.


In [14]:
loaded_model = joblib.load(
    "house_price_model.pkl"
)

prediction = loaded_model.predict(X_test)

print("Sample Predictions:")
print(prediction)

Sample Predictions:
[ 413830.56   900706.98   720171.5    624682.425  335775.62   718678.525
  442492.19  1098479.74   538881.11   469449.4    942195.83   867321.365
  858810.975  800610.85   348828.02   745485.465  760167.245  391696.89
  545650.44   703617.395  361555.735  926911.365 1016637.695  847080.955
  873426.365  288305.695  681320.945  363045.85   833855.015  420159.82
  344666.72   476126.88   484047.555  362934.835 1144903.465 1023134.655
  977437.085  481333.01   910535.435  769186.855  929320.275  415753.815
  326768.325  377718.29   635452.62   388148.08   822281.9    497694.425
  340842.815  816183.435  667105.65   746432.15   920277.7    705901.905
  841061.025  582819.52   835759.475  623473.18   707087.085  596038.24
  899042.275  928842.975  531422.905  701558.095 1116786.095  999289.66
 1049690.335  559246.77   629730.99  1002237.57   702417.835  798161.4
  501535.085  295703.29   767555.9    231019.12  1064930.775 1068668.695
  714043.84   708247.025  801067.61  

In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

prediction = best_model.predict(X_test)

mae = mean_absolute_error(y_test, prediction)
mse = mean_squared_error(y_test, prediction)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, prediction)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R² Score:", r2)

MAE : 24228.955664999998
MSE : 912415997.260015
RMSE: 30206.224478739725
R² Score: 0.9843024222351482


In [16]:
print(X.columns.tolist())

['Area_sqft', 'Bedrooms', 'Bathrooms', 'Floors', 'House_Age', 'Garage', 'Parking', 'Location', 'Furnished', 'Garden', 'Condition', 'Distance_to_City_km']
